# Data Ingest

Loads raw property datasets (PPD, ONSPD, boundaries, amenities, deprivation, etc.) and saves cleaned parquet files for downstream notebooks.

## Setup

In [ ]:
from pathlib import Path
import sys
import numpy as np
import re

from io import StringIO

import pandas as pd
import geopandas as gpd

# Ensure project root is on the Python path
PROJECT_ROOT = Path("..")
PROJECT_ROOT = PROJECT_ROOT.resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from common import io_utils, geo_utils

RAW = Path("../data/raw")
INTERIM = Path("../data/interim")
PROCESSED = Path("../data/processed")
REPORTS = Path("../../reports/tables")

for path in [RAW, INTERIM, PROCESSED, REPORTS]:
    path.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw path:", RAW.resolve())
print("Interim path:", INTERIM.resolve())

Project root: /Users/umar/Documents/Northeastern/Dissertation
Raw path: /Users/umar/Documents/Northeastern/Dissertation/property/data/raw
Interim path: /Users/umar/Documents/Northeastern/Dissertation/property/data/interim


### Snapshot log

| Dataset | Snapshot / Download Date | Notes |
|---------|-----|-------|
| PPD | | |
| ONSPD | | |
| LA Boundaries | | |
| Broadband | | |
| NaPTAN | | |
| OSM POIs | | |
| IMD 2019 | | |
| Households 2021 | | |
| Inside Airbnb | | |

## HM Land Registry Price Paid Data

Loads PPD transactions, cleans dates and postcodes, saves to parquet.

In [6]:
ppd_csv_path = RAW / "ppd" / "ppd_full.csv"
ppd_output   = INTERIM / "ppd_raw.parquet"

# Prefer processed parquet if it exists (much smaller than raw CSV: ~331MB vs 5GB)
if ppd_output.exists():
    print(f"Using existing processed PPD: {ppd_output}")
    print(f"  (To regenerate from CSV, delete this file and run again)")
    ppd = pd.read_parquet(ppd_output)
    print(f"Loaded {len(ppd):,} rows from processed parquet")

elif ppd_csv_path.exists():
    print(f"Reading PPD from {ppd_csv_path} ...")
    print("  (This may take several minutes for the full 5GB file)")
    chunks = pd.read_csv(
        ppd_csv_path,
        header=None,
        usecols=[1, 2, 3, 4, 5, 6],   # only the columns we need
        chunksize=1_000_000,
        low_memory=False
    )

    out = []
    for ch in chunks:
        # Rename by position (safe regardless of how pandas labels them)
        ch.columns = ["price","date","postcode","property_type","old_new","duration"]

        # Clean/parse
        # Dates in PPD are usually ISO-like; be forgiving and set dayfirst=False
        ch["date"] = pd.to_datetime(ch["date"], errors="coerce", utc=False)
        # Uppercase/normalise postcode now (helps later join to ONSPD)
        ch["postcode"] = (ch["postcode"]
                          .astype(str)
                          .str.upper()
                          .str.replace(r"\s+", " ", regex=True)
                          .str.strip())

        out.append(ch)

    ppd = pd.concat(out, ignore_index=True)
    ppd.to_parquet(ppd_output, index=False)
    print(f"Saved {len(ppd):,} rows → {ppd_output}")

else:
    print("PPD file not found.")
    print("  Option 1: Place 'ppd_full.csv' under property/data/raw/ppd/")
    print("  Option 2: Download from: https://www.gov.uk/government/statistical-data-sets/price-paid-data-downloads")
    print("  Option 3: If we have the processed parquet file, place it at:", ppd_output)


Using existing processed PPD: ../data/interim/ppd_raw.parquet
  (To regenerate from CSV, delete this file and run again)
Loaded 30,630,140 rows from processed parquet


## ONS Postcode Directory

Maps postcodes to local authority codes.

In [9]:
onspd_csv_path = RAW / "onspd" / "ONSPD_AUG_2025_UK.csv"
onspd_output   = INTERIM / "onspd_postcode_to_la.parquet"

def normalize_postcode(s: pd.Series) -> pd.Series:
    # Uppercase, collapse any whitespace to a single space, trim
    return (s.astype(str)
             .str.upper()
             .str.replace(r"\s+", " ", regex=True)
             .str.strip())

# Prefer processed parquet if it exists (much smaller than raw CSV: ~50MB vs 1.3GB)
if onspd_output.exists():
    print(f"Using existing processed ONSPD: {onspd_output}")
    print(f"  (To regenerate from CSV, delete this file and run again)")
    df = pd.read_parquet(onspd_output)
    print(f"Loaded {len(df):,} postcode→LA mappings from processed parquet")

elif onspd_csv_path.exists():
    # Read only what is needed; include dointr to pick the latest record
    usecols = ["pcds", "lad25cd", "dointr"] 
    df = pd.read_csv(onspd_csv_path, usecols=usecols, dtype="string", low_memory=False)

    # Rename to standard names
    df = df.rename(columns={"pcds": "postcode", "lad25cd": "LA_code"})

    # Normalise postcode the SAME way as PPD (single space, uppercase)
    df["postcode"] = normalize_postcode(df["postcode"])

    # Keep the latest record per postcode (handles historical duplicates)
    df["dointr"] = pd.to_datetime(df["dointr"], errors="coerce")
    df = df.sort_values(["postcode", "dointr"]).groupby("postcode", as_index=False).tail(1)

    # Clean / filter: drop missing LA, keep England & Wales only (E… / W…)
    df = df.dropna(subset=["LA_code"])
    df = df[df["LA_code"].str.startswith(("E", "W"))]
    df = df[["postcode", "LA_code"]].drop_duplicates().reset_index(drop=True)

    # Save
    df.to_parquet(onspd_output, index=False)
    print(f"Saved postcode→LA lookup: {len(df):,} rows → {onspd_output}")
else:
    print("ONSPD file not found.")
    print("  Option 1: Place 'ONSPD_AUG_2025_UK.csv' under property/data/raw/onspd/")
    print("  Option 2: Download from: https://geoportal.statistics.gov.uk/datasets/ons-postcode-directory-august-2025")
    print("  Option 3: If the processed parquet file is there , place it at:", onspd_output)

Using existing processed ONSPD: ../data/interim/onspd_postcode_to_la.parquet
  (To regenerate from CSV, delete this file and run again)
Loaded 2,403,580 postcode→LA mappings from processed parquet


## Local Authority Boundaries


In [12]:
la_path = RAW / "boundaries" / "lad_boundaries.gpkg"
la_output = INTERIM / "la_boundaries.parquet"
centroids_output = INTERIM / "la_centroids_area.parquet"

if la_path.exists():
    gdf = gpd.read_file(la_path)

    # Find code/name columns across vintages
    code_col = next((c for c in gdf.columns if c.upper().startswith("LAD") and c.upper().endswith("CD")), None)
    name_col = next((c for c in gdf.columns if c.upper().startswith("LAD") and c.upper().endswith("NM")), None)
    if not code_col or not name_col:
        raise ValueError(f"Could not find LAD code/name columns in {la_path.name}. "
                         f"Available columns: {list(gdf.columns)}")

    gdf = gdf[[code_col, name_col, "geometry"]].rename(
        columns={code_col: "LA_code", name_col: "LA_name"}
    )

    # Store boundaries in WGS84 for general use
    gdf = gdf.to_crs(4326)
    gdf.to_parquet(la_output, index=False)
    print(f"Saved LA boundaries ({len(gdf)} areas) → {la_output}")

    # Compute centroid (in meters) and area_km2 safely
    # OSGB36 (EPSG:27700) for Great Britain; if that fails, fall back to 3857
    try:
        gdf_m = gdf.to_crs(27700)
    except Exception:
        gdf_m = gdf.to_crs(3857)

    centroids_m = gdf_m.geometry.centroid
    area_km2 = gdf_m.geometry.area / 1e6

    # bring centroids back to 4326 for saving as lon/lat
    centroids_ll = gpd.GeoSeries(centroids_m, crs=gdf_m.crs).to_crs(4326)
    metrics = pd.DataFrame({
        "LA_code": gdf["LA_code"].values,
        "LA_name": gdf["LA_name"].values,
        "centroid_lon": centroids_ll.x.values,
        "centroid_lat": centroids_ll.y.values,
        "area_km2": area_km2.values
    })
    metrics.to_parquet(centroids_output, index=False)
    print(f"Saved LA centroid/area stats → {centroids_output}")

else:
    print("Boundary file not found. Place the GPKG at property/data/raw/boundaries/ (e.g., 'lad_boundaries.gpkg').")

Saved LA boundaries (361 areas) → ../data/interim/la_boundaries.parquet
Saved LA centroid/area stats → ../data/interim/la_centroids_area.parquet


## Broadband Coverage

Loads gigabit broadband availability data.

In [15]:
broadband_path = RAW / "broadband" / "gigabit-capable-broadband-map-data.csv"
out_path = INTERIM / "broadband_la.parquet"
assert broadband_path.exists(), f"File not found: {broadband_path}"

def _detect_header_and_sep(p: Path, enc="utf-8-sig"):
    with open(p, "r", encoding=enc, errors="ignore") as f:
        lines = [ln.rstrip("\r\n") for ln in f.readlines()[:300] if ln.strip()]
    tokens = ("areacd","area code","areanm","geography","value","time","period")
    hdr_idx = next((i for i, ln in enumerate(lines) if any(t in ln.lower() for t in tokens)), 0)
    hdr = lines[hdr_idx]
    counts = {",": hdr.count(","), "\t": hdr.count("\t"), ";": hdr.count(";"), "|": hdr.count("|")}
    sep = max(counts, key=counts.get) if any(counts.values()) else ","
    return hdr_idx, sep

# Use the renamed var below
hdr_idx, sep = _detect_header_and_sep(broadband_path)
df = pd.read_csv(broadband_path, header=hdr_idx, sep=sep, engine="python", encoding="utf-8-sig")

# Column pickers
cols = list(df.columns)
low = {c.lower().strip(): c for c in cols}
def pick(cands, fuzzy=()):
    for c in cands:
        if c in cols: return c
        if c.lower() in low: return low[c.lower()]
    for c in cols:
        cl = c.lower().strip()
        if any(tok in cl for tok in fuzzy): return c
    return None

code_col = pick(["AREACD","Area Code","LAD25CD","LAD24CD","LAD23CD"], fuzzy=("areacd","area code","lad"))
geo_col  = pick(["Geography"], ("geography",))
time_col = pick(["Time","Period","Date"], ("time","period","date"))
val_col  = pick(["Value","Gigabit %","Percentage"], ("value","gigabit","percent"))

if code_col is None or val_col is None:
    raise ValueError(f"Could not find area code/value columns. Columns: {cols}")

# Keep Local Authority rows (if Geography present)
if geo_col:
    df = df[df[geo_col].astype(str).str.contains("Local Authority", case=False, na=False)].copy()

# If multiple periods, keep latest per area
if time_col:
    t = pd.to_datetime(df[time_col], errors="coerce")
    if t.notna().any():
        df = (df.assign(__t__=t)
                .sort_values([code_col,"__t__"])
                .groupby(code_col, as_index=False)
                .tail(1)
                .drop(columns="__t__"))
    else:
        df["__p__"] = pd.to_numeric(df[time_col], errors="coerce")
        df = (df.sort_values([code_col,"__p__"])
                .groupby(code_col, as_index=False)
                .tail(1)
                .drop(columns="__p__"))

# Convert percentage to 0..1 and build output
gigabit_pct = pd.to_numeric(df[val_col], errors="coerce") / 100.0
out_df = (pd.DataFrame({
            "LA_code": df[code_col].astype(str).str.strip(),
            "gigabit_pct": gigabit_pct
        })
        .dropna(subset=["gigabit_pct"]))

# England/Wales only
out_df = out_df[out_df["LA_code"].str.startswith(("E","W"), na=False)].drop_duplicates("LA_code", keep="last")

out_df.to_parquet(out_path, index=False)
print(f"Saved {len(out_df)} LAs → {out_path}")
print(out_df.head())

Saved 318 LAs → ../data/interim/broadband_la.parquet
     LA_code  gigabit_pct
0  E06000001        0.959
1  E06000002        0.960
2  E06000003        0.908
3  E06000004        0.928
4  E06000005        0.919


## NaPTAN Public Transport Stops

Processes transport stop locations for accessibility features.

In [18]:
naptan_path = RAW / "naptan" / "Stops.csv"
naptan_output = INTERIM / "naptan_stops.parquet"

# Debug: print paths to help diagnose
print(f"RAW path: {RAW}")
print(f"RAW resolved: {RAW.resolve()}")
print(f"NaPTAN path: {naptan_path}")
print(f"NaPTAN resolved: {naptan_path.resolve()}")
print(f"File exists: {naptan_path.exists()}")

if naptan_path.exists():
    cols = ["ATCOCode", "StopType", "Latitude", "Longitude", "Status"]
    naptan = pd.read_csv(naptan_path, usecols=cols)
    naptan = naptan.rename(columns={
        "ATCOCode": "atco_code",
        "StopType": "stop_type",
        "Latitude": "latitude",
        "Longitude": "longitude",
        "Status": "status",
    })
    naptan = naptan[naptan["status"].str.upper() == "ACTIVE"]
    naptan["is_tube"] = naptan["stop_type"].eq("MET") | naptan["atco_code"].str.startswith("940G")
    naptan["is_rail"] = naptan["stop_type"].eq("RLY") | naptan["atco_code"].str.startswith("910G")
    gdf = geo_utils.make_point_gdf(naptan, lat_col="latitude", lon_col="longitude")
    gdf.to_parquet(naptan_output, index=False)
    print(f"Saved NaPTAN stops → {naptan_output}")
else:
    print("Stops.csv not found. Place NaPTAN extract under property/data/raw/naptan/.")

RAW path: ../data/raw
RAW resolved: /Users/umar/Documents/Northeastern/Dissertation/property/data/raw
NaPTAN path: ../data/raw/naptan/Stops.csv
NaPTAN resolved: /Users/umar/Documents/Northeastern/Dissertation/property/data/raw/naptan/Stops.csv
File exists: True
Saved NaPTAN stops → ../data/interim/naptan_stops.parquet


## OpenStreetMap Amenities

Loads fast-food and supermarket POIs from GeoJSON exports for density calculations.

In [21]:
osm_dir = RAW / "amenities"
out_fast = INTERIM / "amenities_fast_food.parquet"
out_super = INTERIM / "amenities_supermarket.parquet"

if not osm_dir.exists():
    raise FileNotFoundError(
        f"OSM export folder not found: {osm_dir}\n"
        "Place Overpass/Overpass Turbo GeoJSON files under ../data/raw/amenities/"
    )

geojson_files = sorted(osm_dir.glob("*.geojson"))
if not geojson_files:
    raise FileNotFoundError(f"No .geojson files found in {osm_dir}")

ff_frames, sm_frames = [], []

for path in geojson_files:
    gdf = gpd.read_file(path)

    # Ensure WGS84
    if getattr(gdf, "crs", None) is None:
        gdf = gdf.set_crs(4326)
    else:
        gdf = gdf.to_crs(4326)

    # Standardise columns that we might use later
    for c in ("amenity", "shop", "name", "brand"):
        if c not in gdf.columns:
            gdf[c] = pd.NA
    gdf = gdf[["amenity", "shop", "name", "brand", "geometry"]].copy()

    # Convert polygons/lines to centroids so we have points for joins
    gdf["geometry"] = gdf.geometry.centroid

    # Split by category
    ff = gdf[gdf["amenity"].eq("fast_food")].copy()
    sm = gdf[gdf["shop"].eq("supermarket")].copy()

    # De-duplicate identical coords per tag to avoid mall overcounts
    for df_ in (ff, sm):
        if df_.empty:
            continue
        df_["__x__"] = df_.geometry.x.round(6)
        df_["__y__"] = df_.geometry.y.round(6)
        df_.drop_duplicates(["__x__","__y__","amenity","shop"], inplace=True)
        df_.drop(columns=["__x__","__y__"], inplace=True)

    if not ff.empty: ff_frames.append(ff)
    if not sm.empty: sm_frames.append(sm)

# Concatenate & save even if one category is empty
ff_all = gpd.GeoDataFrame(pd.concat(ff_frames, ignore_index=True), crs=4326) if ff_frames else gpd.GeoDataFrame(columns=["amenity","shop","name","brand","geometry"], crs=4326)
sm_all = gpd.GeoDataFrame(pd.concat(sm_frames, ignore_index=True), crs=4326) if sm_frames else gpd.GeoDataFrame(columns=["amenity","shop","name","brand","geometry"], crs=4326)

ff_all.to_parquet(out_fast, index=False)
sm_all.to_parquet(out_super, index=False)

print(f"Saved {len(ff_all)} fast-food POIs → {out_fast}")
print(f"Saved {len(sm_all)} supermarket POIs → {out_super}")

/var/folders/m2/10p9b8xj3gg5xtr6q8r54cs40000gn/T/ipykernel_90304/539620341.py:33: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf["geometry"] = gdf.geometry.centroid


Saved 45676 fast-food POIs → ../data/interim/amenities_fast_food.parquet
Saved 10238 supermarket POIs → ../data/interim/amenities_supermarket.parquet


/var/folders/m2/10p9b8xj3gg5xtr6q8r54cs40000gn/T/ipykernel_90304/539620341.py:33: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf["geometry"] = gdf.geometry.centroid
/var/folders/m2/10p9b8xj3gg5xtr6q8r54cs40000gn/T/ipykernel_90304/539620341.py:33: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf["geometry"] = gdf.geometry.centroid


## Indices of Multiple Deprivation

Loads IMD scores at LSOA level.

In [24]:
imd_path = RAW / "imd" / "IMD_deprivation.csv"
imd_output = INTERIM / "imd_lsoa.parquet"

if imd_path.exists():
    imd = pd.read_csv(imd_path)
    imd = imd.rename(columns={
        "LSOA code (2011)": "LSOA_code",
        "Local Authority District code (2019)": "LA_code",
        "Index of Multiple Deprivation (IMD) Score": "imd_score",
    })
    imd = imd[["LSOA_code", "LA_code", "imd_score"]]
    io_utils.save_parquet(imd, imd_output)
    print(f"Saved IMD table → {imd_output}")
else:
    print("IMD file not found. Update path/filename above.")

Saved IMD table → ../data/interim/imd_lsoa.parquet


## Households

Loads 2021 Census household counts per local authority.

In [27]:
in_path  = RAW / "households" / "TS041_households.csv"
out_path = INTERIM / "households_la.parquet"

# 1) Read raw lines and find the true header row
with open(in_path, "r", encoding="utf-8-sig", errors="ignore") as f:
    lines = [ln.rstrip("\r\n") for ln in f if ln.strip()]

# Find the first line that looks like the header (has 'mnemonic' and a year like '2021')
header_idx = None
for i, ln in enumerate(lines):
    low = ln.lower()
    if ("mnemonic" in low) and ("2021" in low or "obs_value" in low or "value" in low):
        header_idx = i
        break
if header_idx is None:
    raise ValueError("Could not locate header row (looked for a line with 'mnemonic' and '2021').")

# 2) Read the body from that line onward as a CSV
body = "\n".join(lines[header_idx:])
df = pd.read_csv(StringIO(body), dtype=object)

# 3) Pick columns
cols = {c.lower().strip(): c for c in df.columns}

# name column usually contains 'local authority'
name_col = next((v for k, v in cols.items() if "local authority" in k), None)
# code column is 'mnemonic'
code_col = next((v for k, v in cols.items() if "mnemonic" in k), None)
# value column could be literal '2021', or 'obs_value'/'value'
value_col = None
for key in ["2021", "obs_value", "value"]:
    if key in cols:
        value_col = cols[key]
        break
# also try any column that looks numeric-year-like
if value_col is None:
    for k, v in cols.items():
        if k.isdigit():
            value_col = v
            break

if not all([name_col, code_col, value_col]):
    raise ValueError(f"Could not find expected columns. Found: {list(df.columns)}")

# 4) Clean and normalise
out = (df[[code_col, name_col, value_col]].copy()
         .rename(columns={
             code_col: "LA_code",
             name_col: "LA_name",
             value_col: "households_2021"
         }))

# Strip, keep England/Wales LAs, coerce numeric
out["LA_code"] = out["LA_code"].astype(str).str.strip()
out["LA_name"] = out["LA_name"].astype(str).str.strip()
out = out[out["LA_code"].str.match(r"^[EW]\d{8}$", na=False)]

out["households_2021"] = pd.to_numeric(out["households_2021"], errors="coerce")
out = out.dropna(subset=["households_2021"]).drop_duplicates("LA_code", keep="last")

# 5) Save
out.to_parquet(out_path, index=False)
print(f"Saved households table ({len(out)} LAs) → {out_path}")
print(out.head(5))

Saved households table (318 LAs) → ../data/interim/households_la.parquet
     LA_code    LA_name  households_2021
0  E07000200    Babergh            40200
1  E07000066   Basildon            76362
2  E06000055    Bedford            74950
3  E07000067  Braintree            64986
4  E07000143  Breckland            60359


## InsideAirbnb Listings

Loads Airbnb listing data for selected cities.

In [30]:
airbnb_dir   = RAW / "airbnb"
flat_out     = INTERIM / "insideairbnb_listings_city.parquet"
geo_out      = INTERIM / "insideairbnb_listings_city_geo.parquet"

def _read_airbnb_file(path: Path) -> pd.DataFrame:
    # Standard columns present in normal InsideAirbnb listings exports
    wanted = ["id", "last_scraped", "latitude", "longitude", "availability_365"]
    df = pd.read_csv(path, low_memory=False)
    # Keep only what we need (ignore if any are missing)
    keep = [c for c in wanted if c in df.columns]
    if len(keep) < 4:
        raise ValueError(f"File missing expected columns: {path.name} (found {list(df.columns)[:10]}...)")
    df = df[keep].copy()
    df["id"] = df["id"].astype(str)
    # Infer city from filename prefix before first space/underscore
    stem = path.stem  # e.g., "london-2025-10-...listings.csv"
    city = stem.split()[0].split("_")[0].split("-")[0].lower()
    df["city"] = city
    return df

if airbnb_dir.exists():
    files = sorted(list(airbnb_dir.glob("*.csv.gz")) + list(airbnb_dir.glob("*.csv")))
    if not files:
        print(f"No InsideAirbnb CSVs found in {airbnb_dir}")
    else:
        frames = []
        for p in files:
            try:
                frames.append(_read_airbnb_file(p))
            except Exception as e:
                print(f"Skipping {p.name}: {e}")

        if not frames:
            print("No valid InsideAirbnb files parsed.")
        else:
            airbnb = pd.concat(frames, ignore_index=True)

            # Clean timestamps and coords
            airbnb["last_scraped"] = pd.to_datetime(airbnb["last_scraped"], errors="coerce")
            airbnb = airbnb.dropna(subset=["last_scraped", "latitude", "longitude"])

            # Keep the latest record per listing id
            airbnb = (
                airbnb.sort_values(["id", "last_scraped"])
                      .groupby("id", as_index=False)
                      .tail(1)
            )

            # Monthly tag for later LA×month aggregation
            airbnb["snapshot_month"] = airbnb["last_scraped"].dt.to_period("M").dt.to_timestamp("M")

            # Save flat table
            airbnb.to_parquet(flat_out, index=False)

            # Optional: GeoParquet for spatial previews
            airbnb_gdf = geo_utils.make_point_gdf(airbnb, lat_col="latitude", lon_col="longitude")
            airbnb_gdf.to_parquet(geo_out, index=False)

            n_listings = len(airbnb)
            n_cities   = airbnb["city"].nunique()
            t_min, t_max = airbnb["last_scraped"].min(), airbnb["last_scraped"].max()
            print(f"Saved InsideAirbnb listings → {flat_out} ({n_listings:,} listings, {n_cities} cities, {t_min.date()}–{t_max.date()})")
            print(f"Saved GeoParquet → {geo_out}")
else:
    print("InsideAirbnb directory not found. Place listings CSVs under property/data/raw/airbnb/.")


Saved InsideAirbnb listings → ../data/interim/insideairbnb_listings_city.parquet (107,133 listings, 3 cities, 2025-09-14–2025-09-26)
Saved GeoParquet → ../data/interim/insideairbnb_listings_city_geo.parquet


## ONS Private Rental Index

Builds monthly rent indices by local authority from PRMS snapshot and England monthly index.


In [32]:
#Setup & paths
from pathlib import Path
import pandas as pd
import numpy as np
import re

RAW_RENTAL = Path("../data/raw/rental")
INTERIM = Path("../data/interim")
INTERIM.mkdir(parents=True, exist_ok=True)

PRMS_RAW = next((p for p in RAW_RENTAL.glob("privaterentalmarketstatistics231220*.*")), None)
IPHRP_RAW = next((p for p in RAW_RENTAL.glob("priceindexofprivaterentsukmonthlypricestatistics*.*")), None)

print("PRMS file:", PRMS_RAW)
print("IPHRP file:", IPHRP_RAW)

PRMS file: ../data/raw/rental/privaterentalmarketstatistics231220.xls
IPHRP file: ../data/raw/rental/priceindexofprivaterentsukmonthlypricestatistics.xlsx


### Name Normaliser and LA Lookup


In [36]:
def norm_name(s: str) -> str:
    s = str(s).lower()
    s = re.sub(r"\s*\(met county\)", "", s)
    s = re.sub(r"\s+ua\s*$", "", s)
    s = re.sub(r"[-&/,']", " ", s)
    s = re.sub(r"\s*(borough|city|county)\s+of\s*", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

# Minimal LA lookup used across notebooks (LA_code, LA_name)
panel_lookup_path = INTERIM / "panel_min_lookup.parquet"

if panel_lookup_path.exists():
    la_lookup = pd.read_parquet(panel_lookup_path)[["LA_code","LA_name"]].drop_duplicates()
    print(f"LA lookup loaded → {panel_lookup_path} ({len(la_lookup)} LAs)")
else:
    # If la_boundaries.parquet is there 
    boundaries_path = INTERIM / "la_boundaries.parquet"
    if not boundaries_path.exists():
        raise FileNotFoundError("Need la_boundaries.parquet to create panel_min_lookup.parquet")
    la_lookup = pd.read_parquet(boundaries_path)[["LA_code","LA_name"]].drop_duplicates()
    la_lookup.to_parquet(panel_lookup_path, index=False)
    print(f"Created LA lookup → {panel_lookup_path} ({len(la_lookup)} LAs)")

# Add normalised names
la_lookup = la_lookup.assign(_n = la_lookup["LA_name"].map(norm_name))


LA lookup loaded → ../data/interim/panel_min_lookup.parquet (361 LAs)


### Build PRMS Table 2.7

In [39]:
#Parse PRMS Table 2.7 (the sheet has header on row 6) and save with LA_code

assert PRMS_RAW is not None, "Place the PRMS file in ../data/raw/rental/"

import re

def _norm_cols(cols):
    return [re.sub(r"\s+", " ", str(c).strip()) for c in cols]

def _norm_name(s: str) -> str:
    s = str(s).lower()
    s = re.sub(r"\s*\(met county\)", "", s)
    s = re.sub(r"\s+ua\s*$", "", s)
    s = re.sub(r"[-&/,']", " ", s)
    s = re.sub(r"\s*(borough|city|county)\s+of\s*", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def _read_prms_table27_header6(path):
    # Prefer sheets named like Table 2.7
    xls = pd.ExcelFile(path)
    sheet_candidates = [s for s in xls.sheet_names if "2.7" in s.lower()] or xls.sheet_names
    last_err = None
    for sh in sheet_candidates:
        try:
            df = pd.read_excel(path, sheet_name=sh, header=6, dtype=object)
            df.columns = _norm_cols(df.columns)
            # We expect these columns to exist with these exact display names
            required = {"Area Code1", "Area", "Median"}
            if not required.issubset(set(df.columns)):
                raise ValueError(f"Sheet '{sh}' with header=6 lacks columns {required - set(df.columns)}. Has: {list(df.columns)}")
            out = df[["Area Code1", "Area", "Median"]].rename(
                columns={"Area Code1":"LA_code", "Area":"LA_name", "Median":"median_rent"}
            ).copy()

            # Clean rows
            out = out.dropna(subset=["LA_code","LA_name"])
            # Keep only proper English LAs (districts/unitaries/london boroughs/met districts)
            out["LA_code"] = out["LA_code"].astype(str).str.strip()
            mask_la = out["LA_code"].str.startswith(("E060","E070","E080","E090"))
            out = out.loc[mask_la].copy()

            # Median as number
            out["median_rent"] = (
                out["median_rent"].astype(str)
                .str.replace(",", "", regex=False)
                .str.replace("£", "", regex=False)
                .str.extract(r"([0-9]+(?:\.[0-9]+)?)", expand=False)
            )
            out["median_rent"] = pd.to_numeric(out["median_rent"], errors="coerce")
            out = out.dropna(subset=["median_rent"])

            # Final tidy
            out["LA_name"] = out["LA_name"].astype(str).str.strip()
            out = out.drop_duplicates("LA_code")

            print(f"[PRMS] Parsed '{sh}' with header row 6 → {len(out)} LAs")
            return out
        except Exception as e:
            last_err = e
            continue
    raise last_err if last_err else RuntimeError("Failed to parse PRMS Table 2.7")

# 1) Read PRMS using the known layout 
prms2 = _read_prms_table27_header6(PRMS_RAW)

# 2) (Optional) sanity-print a few rows
display(prms2.head(12))

# 3) Save to interim for downstream notebooks
out_prms = INTERIM / "prms_table_2_7_la_medians.parquet"
prms2.to_parquet(out_prms, index=False)
print(f"Saved PRMS Table 2.7 with codes → {out_prms} ({len(prms2)} LAs)")

# 4) Quick coverage note vs panel (if lookup available)
panel_lookup_path = INTERIM / "panel_min_lookup.parquet"
if panel_lookup_path.exists():
    la_lookup = pd.read_parquet(panel_lookup_path)[["LA_code","LA_name"]].drop_duplicates()
    cov = prms2["LA_code"].isin(la_lookup["LA_code"]).mean()*100
    print(f"[COVERAGE] PRMS LAs found in panel lookup: {cov:.1f}%")
else:
    print("[COVERAGE] No panel_min_lookup.parquet yet—skipping coverage check.")


[PRMS] Parsed 'Table2.7' with header row 6 → 314 LAs


,LA_code,LA_name,median_rent
2,E06000047,County Durham UA,498
3,E06000005,Darlington UA,505
4,E06000001,Hartlepool UA,505
5,E06000002,Middlesbrough UA,525
6,E06000057,Northumberland UA,550
7,E06000003,Redcar and Cleveland UA,525
8,E06000004,Stockton-on-Tees UA,575
10,E08000037,Gateshead,595
11,E08000021,Newcastle upon Tyne,795
12,E08000022,North Tyneside,600


Saved PRMS Table 2.7 with codes → ../data/interim/prms_table_2_7_la_medians.parquet (314 LAs)
[COVERAGE] PRMS LAs found in your panel lookup: 91.7%


### Clean England Monthly Index

In [42]:
# Build iphrp_england_monthly.parquet

assert IPHRP_RAW is not None, "Set IPHRP_RAW to the downloaded 'price index of private rents' file."

def _find_header_row(df0: pd.DataFrame, must_have=("time period","area name","index")):
    """
    Given a DataFrame read with header=None, try to find the row index that
    contains the real column headers (e.g. 'Time period', 'Area name', 'Index').
    Returns the integer row index, or None if not found.
    """
    # Make everything stringy and lower
    lower = df0.astype(str).applymap(lambda x: x.strip().lower())
    for r in range(min(60, len(lower))):  # scan first ~60 rows
        row_vals = set(lower.iloc[r, :].tolist())
        if all(any(needle in v for v in row_vals) for needle in must_have):
            return r
    return None

def _read_iphrp_any(path: Path) -> pd.DataFrame:
    """
    Read the workbook (Excel/CSV), find a sheet that contains a proper header row,
    return a tidy DataFrame with at least: time_period, area_code/area_name, index.
    """
    if path.suffix.lower() in (".xlsx", ".xls"):
        xls = pd.ExcelFile(path)
        # Prefer a sheet containing 'table 1' in its name; otherwise scan all.
        sheet_order = sorted(
            xls.sheet_names,
            key=lambda s: (0 if re.search(r"\btable\s*1\b", s, flags=re.I) else 1, s)
        )
        tried = []
        for sh in sheet_order:
            df0 = pd.read_excel(path, sheet_name=sh, header=None)
            hdr = _find_header_row(df0)
            tried.append((sh, hdr))
            if hdr is None:
                continue
            df = pd.read_excel(path, sheet_name=sh, header=hdr)
            # normalise cols
            df.columns = [str(c).strip().lower().replace(" ", "_") for c in df.columns]
            return df
        # If we get here, show a small debug print then fail
        print("[DEBUG] Tried sheets and header rows:", tried[:6], "...")
        raise ValueError("Could not locate a row with headers (e.g., 'Time period', 'Area name', 'Index').")
    else:
        # CSV fallback
        df = pd.read_csv(path, header=None)
        hdr = _find_header_row(df)
        if hdr is None:
            raise ValueError("CSV did not contain a recognizable header row.")
        df = pd.read_csv(path, header=hdr)
        df.columns = [str(c).strip().lower().replace(" ", "_") for c in df.columns]
        return df

def _parse_month(series: pd.Series) -> pd.Series:
    """
    Parse time period values into month-end timestamps.
    Handles forms like '2020-01', 'Jan-2020', '2020 JAN', '2020M01'.
    """
    s = pd.to_datetime(series, errors="coerce")
    if s.isna().mean() > 0.5:
        # Try Period(M)
        try:
            s = pd.PeriodIndex(series.astype(str), freq="M").to_timestamp("M")
        except Exception:
            pass
    return s + pd.offsets.MonthEnd(0)

# Load the messy workbook and get a proper table
df = _read_iphrp_any(IPHRP_RAW)

# Some ONS sheets have more columns; we only need time period + an England-wide index/value
# Find candidate time column
time_col = None
for c in df.columns:
    if any(k in c for k in ("time_period", "time", "month", "date")):
        time_col = c
        break
if time_col is None:
    raise ValueError(f"Could not find a time column. Got columns: {df.columns.tolist()}")

# Find England rows. We’ll try by code, then names in a few possible columns.
def _is_england(frame: pd.DataFrame) -> pd.Series:
    cols = frame.columns
    s = pd.Series(False, index=frame.index)
    if "area_code" in cols:
        s |= frame["area_code"].astype(str).str.upper().eq("E92000001")
    for name_col in ("area_name", "region_or_country_name", "country"):
        if name_col in cols:
            s |= frame[name_col].astype(str).str.contains(r"\bengland\b", case=False, na=False)
    return s

eng_df = df[_is_england(df)].copy()
if eng_df.empty:
    # last resort: any column with 'england' anywhere in the row
    mask = pd.Series(False, index=df.index)
    for c in df.columns:
        mask |= df[c].astype(str).str.contains(r"\bengland\b", case=False, na=False)
    eng_df = df[mask].copy()

if eng_df.empty:
    print("[DEBUG] Columns present:", df.columns.tolist())
    raise ValueError("Could not isolate England rows. Check the sheet layout or rename clues.")

# Choose the index/value column:
# Prefer a column literally called 'index', else the last numeric column
val_col = None
if "index" in eng_df.columns:
    val_col = "index"
else:
    num_cols = [c for c in eng_df.columns if pd.api.types.is_numeric_dtype(eng_df[c])]
    if not num_cols:
        raise ValueError("No numeric index/value column found for England rows.")
    val_col = num_cols[-1]

eng = eng_df[[time_col, val_col]].rename(columns={time_col: "month", val_col: "iphrp_england"}).copy()
eng["month"] = _parse_month(eng["month"])
eng["iphrp_england"] = pd.to_numeric(eng["iphrp_england"], errors="coerce")

eng = (eng.dropna(subset=["month", "iphrp_england"])
          .drop_duplicates(subset=["month"])
          .sort_values("month"))

if eng.empty:
    raise ValueError("Parsed England series is empty after cleaning.")

print(f"[IPHRP] England monthly span: {eng['month'].min().date()} → {eng['month'].max().date()} "
      f"({len(eng)} months)")

out_iphrp = INTERIM / "iphrp_england_monthly.parquet"
eng.to_parquet(out_iphrp, index=False)
print(f"Saved England monthly index → {out_iphrp}")
display(eng.head())


/var/folders/m2/10p9b8xj3gg5xtr6q8r54cs40000gn/T/ipykernel_90304/1332659911.py:12: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  lower = df0.astype(str).applymap(lambda x: x.strip().lower())


[IPHRP] England monthly span: 2015-01-31 → 2025-09-30 (129 months)
Saved England monthly index → ../data/interim/iphrp_england_monthly.parquet


,month,iphrp_england
258,2015-01-31,81.458496
259,2015-02-28,81.608616
260,2015-03-31,81.870067
261,2015-04-30,82.212893
262,2015-05-31,82.586593


### Build LA×Month Rent Index

In [ ]:
# Build LA-level monthly rent_index.parquet

prms_la = pd.read_parquet(INTERIM / "prms_table_2_7_la_medians.parquet")  # LA_code, median_rent
eng     = pd.read_parquet(INTERIM / "iphrp_england_monthly.parquet")      # month, iphrp_england

# Basic sanity
assert {"LA_code","median_rent"}.issubset(prms_la.columns), prms_la.columns
assert {"month","iphrp_england"}.issubset(eng.columns), eng.columns

# Normalise time column
eng["month"] = pd.to_datetime(eng["month"]) + pd.offsets.MonthEnd(0)
eng = eng.sort_values("month").dropna(subset=["iphrp_england"])
# Rebase to first non-NaN value
base = eng["iphrp_england"].dropna().iloc[0]
eng["scale"] = eng["iphrp_england"] / base

# Keep only usable PRMS rows
prms_la = prms_la.dropna(subset=["LA_code","median_rent"]).copy()
prms_la["median_rent"] = prms_la["median_rent"].astype("float64")
# Guard against non-positive rents (shouldn't happen, but safer)
prms_la.loc[prms_la["median_rent"] <= 0, "median_rent"] = np.nan
prms_la = prms_la.dropna(subset=["median_rent"])

# Cartesian join: all LAs × all months
rent_index = (
    prms_la.assign(_k=1)
           .merge(eng[["month","scale"]].assign(_k=1), on="_k", how="inner")
           .drop(columns="_k")
)

# Construct monthly rent index
rent_index["rent_index"] = (rent_index["median_rent"] * rent_index["scale"]).astype("float32")
rent_index = rent_index[["LA_code","month","rent_index"]].sort_values(["LA_code","month"])

# Diagnostics
n_las = rent_index["LA_code"].nunique()
n_months = rent_index["month"].nunique()
assert n_las == prms_la["LA_code"].nunique(), "Lost some LAs during join"
assert rent_index["rent_index"].gt(0).all(), "Found non-positive rent_index values"

# Save
out_rent = INTERIM / "rent_index.parquet"
rent_index.to_parquet(out_rent, index=False)
print(f"Saved rent_index → {out_rent} ({len(rent_index):,} rows; {n_las} LAs × {n_months} months)")
display(rent_index.head())

Saved rent_index → ../data/interim/rent_index.parquet (40,506 rows; 314 LAs × 129 months)


,LA_code,month,rent_index
258,E06000001,2015-01-31,505.000000
259,E06000001,2015-02-28,505.930664
260,E06000001,2015-03-31,507.551514
261,E06000001,2015-04-30,509.676880
262,E06000001,2015-05-31,511.993622


## Summary

In [48]:
summary_files = [
    INTERIM / "ppd_raw.parquet",
    INTERIM / "onspd_postcode_to_la.parquet",
    INTERIM / "la_boundaries.parquet",
    INTERIM / "la_centroids_area.parquet",
    INTERIM / "broadband_la.parquet",
    INTERIM / "naptan_stops.parquet",
    INTERIM / "amenities_fast_food.parquet",
    INTERIM / "amenities_supermarket.parquet",
    INTERIM / "imd_lsoa.parquet",
    INTERIM / "households_la.parquet",
    INTERIM / "insideairbnb_listings_city.parquet",
    INTERIM / "insideairbnb_listings_city_geo.parquet",
    INTERIM / "prms_table_2_7_la_medians.parquet",
    INTERIM / "panel_min_lookup.parquet",
    INTERIM / "iphrp_england_monthly.parquet",
    INTERIM / "rent_index.parquet",
]
for path in summary_files:
    exists = path.exists()
    print(f"{'OK' if exists else 'MISSING'} {path.name}")

OK ppd_raw.parquet
OK onspd_postcode_to_la.parquet
OK la_boundaries.parquet
OK la_centroids_area.parquet
OK broadband_la.parquet
OK naptan_stops.parquet
OK amenities_fast_food.parquet
OK amenities_supermarket.parquet
OK imd_lsoa.parquet
OK households_la.parquet
OK insideairbnb_listings_city.parquet
OK insideairbnb_listings_city_geo.parquet
OK prms_table_2_7_la_medians.parquet
OK panel_min_lookup.parquet
OK iphrp_england_monthly.parquet
OK rent_index.parquet
